In [47]:
import pandas as pd
#import cudf
import pickle
import matplotlib.pyplot as plt
from matplotlib import figure
import numpy as np
#import cupy as cp
from scipy.io import wavfile
#from scipy import signal, stats
#import peakutils, wfdb, pywt
import csv
import os, statistics
from datetime import datetime
#import heartpy as hp
import json
%matplotlib notebook
# import neurokit2 as nk

In [48]:
# fE4 = 'A04C05_220712-181114'
# fe4 = '220712-181114'
# fhex = 'record_248582'
fE4 = 'A0381C_240420-210454'
fhex = 'record_289634'

# Import E4 CSV files

In [49]:
# read E4 data, clean up with corrected timstamp and store the csv files in dict
def read_E4(filepath):
    # HR data -- started 10 seconds later than other metrics
    hr = pd.read_csv(filepath+str('/HR.csv'), header = None)
    # clean up HR file
    start_time = hr.values[0]
    hr_samp_rate = hr.values[1]
    hr = hr.drop(labels = [0, 1], axis = 0, inplace = False)
    hr['Timestamp'] = list(range(0, len(hr),1))
    hr['Timestamp'] = hr['Timestamp'].apply(lambda x: x/hr_samp_rate+start_time)
    hr['Timestamp'] = hr['Timestamp'].str.get(0)
    hr['Second'] = hr['Timestamp']
    hr = hr.set_index('Timestamp')
    hr['Second'] = hr['Second'].apply(lambda x: x-hr.index[0])
    hr.columns = ['Heart rate', 'Second']
    hr = hr.reset_index(inplace=False)
    
    # EDA data
    eda = pd.read_csv(filepath+str('/EDA.csv'), header = None)
    start_time = eda.values[0]
    eda_samp_rate = eda.values[1]
    eda = eda.drop(labels = [0, 1], axis = 0, inplace = False)
    eda['Timestamp'] = list(range(0, len(eda),1))
    eda['Timestamp'] = eda['Timestamp'].apply(lambda x: x/eda_samp_rate+start_time)
    eda['Timestamp'] = eda['Timestamp'].str.get(0)
    eda['Second'] = eda['Timestamp']
    eda = eda.set_index('Timestamp')
    eda['Second'] = eda['Second'].apply(lambda x: x-eda.index[0])
    eda.columns = ['EDA', 'Second']
    eda = eda.reset_index(inplace=False)

    temp = pd.read_csv(filepath+str('/TEMP.csv'), header = None)
    # clean up TEMP file
    start_time = temp.values[0]
    temp_samp_rate = temp.values[1]
    temp = temp.drop(labels = [0, 1], axis = 0, inplace = False)
    temp['Timestamp'] = list(range(0, len(temp),1))
    temp['Timestamp'] = temp['Timestamp'].apply(lambda x: x/temp_samp_rate+start_time)
    temp['Timestamp'] = temp['Timestamp'].str.get(0)
    temp['Second'] = temp['Timestamp']
    temp = temp.set_index('Timestamp')
    temp['Second'] = temp['Second'].apply(lambda x: x-temp.index[0])
    temp.columns = ['Temp', 'Second']
    temp = temp.reset_index(inplace=False)
    
    ibi = pd.read_csv(filepath+str('/IBI.csv'), header = None) # no correction of timestamp needed
    ibi = ibi.drop(labels = [0, 1], axis = 0, inplace = False)
    ibi.columns = ['Second', 'IBI']
    
    
    bvp = pd.read_csv(filepath+str('/BVP.csv'), header = None)
    start_time = bvp.values[0]
    bvp_samp_rate = bvp.values[1]
    bvp = bvp.drop(labels = [0, 1], axis = 0, inplace = False)
    bvp['Timestamp'] = list(range(0, len(bvp),1))
    bvp['Timestamp'] = bvp['Timestamp'].apply(lambda x: np.round(x/bvp_samp_rate+start_time, 2))
    bvp['Timestamp'] = bvp['Timestamp'].str.get(0)
    bvp['Second'] = bvp['Timestamp']
    bvp = bvp.set_index('Timestamp')
    bvp['Second'] = bvp['Second'].apply(lambda x: x-bvp.index[0])
    bvp.columns = ['BVP', 'Second']
    bvp = bvp.reset_index(inplace=False)
    
    acc = pd.read_csv(filepath+str('/ACC.csv'), header = None)
    start_time = acc.values[0,0]
    acc_samp_rate = acc.values[1,0]
    acc = acc.drop(labels = [0, 1], axis = 0, inplace = False)
    acc['Timestamp'] = list(range(0, len(acc),1))
    acc['Timestamp'] = acc['Timestamp'].apply(lambda x: x/acc_samp_rate+start_time)
    acc['Second'] = acc['Timestamp']
    acc = acc.set_index('Timestamp')
    acc['Second'] = acc['Second'].apply(lambda x: x-acc.index[0])
    acc.columns = ['ACC_X','ACC_Y','ACC_Z','Second']
    acc = acc.reset_index(inplace=False)
    
    data_dict = {'HR':hr, 'EDA':eda, 'TEMP':temp, 'IBI':ibi, 'BVP':bvp,'ACC':acc}
    return data_dict 

In [50]:
filepath=r'/home/maxinehe/Downloads/' + fE4
a=read_E4(filepath)

In [51]:
a

{'HR':          Timestamp  Heart rate  Second
 0     1.713647e+09       49.00     0.0
 1     1.713647e+09       67.00     1.0
 2     1.713647e+09       86.00     2.0
 3     1.713647e+09       78.25     3.0
 4     1.713647e+09       74.80     4.0
 ...            ...         ...     ...
 4868  1.713652e+09       74.37  4868.0
 4869  1.713652e+09       74.33  4869.0
 4870  1.713652e+09       74.25  4870.0
 4871  1.713652e+09       74.12  4871.0
 4872  1.713652e+09       74.03  4872.0
 
 [4873 rows x 3 columns],
 'EDA':           Timestamp       EDA   Second
 0      1.713647e+09  0.000000     0.00
 1      1.713647e+09  0.043563     0.25
 2      1.713647e+09  0.088407     0.50
 3      1.713647e+09  0.085844     0.75
 4      1.713647e+09  0.085844     1.00
 ...             ...       ...      ...
 19513  1.713652e+09  0.075594  4878.25
 19514  1.713652e+09  0.076875  4878.50
 19515  1.713652e+09  0.076875  4878.75
 19516  1.713652e+09  0.078157  4879.00
 19517  1.713652e+09  0.076875  4879.25

# Import Hexoskin files

In [52]:
# read raw ECG file; ECG_I.wav only
# change wavefile pathway
raw_ECG = wavfile.read(r'/home/maxinehe/Downloads/'+fhex+'/ECG_I.wav')
#settings = {}
#settings['fs'] = 256 # sampling rate
raw_ECG = pd.DataFrame(data = raw_ECG[1])
ecg = 0.0064 * raw_ECG #get correct magnitude of ECG
ecg.rename(columns = {0: 'ECG'}, inplace = True)
# Opening JSON file and return it as dictionary
# change file pathway
f = open(r'/home/maxinehe/Downloads/'+fhex+'/info.json')
date_info = json.load(f)

In [53]:
raw_br = wavfile.read(r'/home/maxinehe/Downloads/'+fhex+'/breathing_rate.wav')
raw_br = pd.DataFrame(data = raw_br[1])
br = 1.0000 * raw_br
br.rename(columns = {0: 'breathing_rate'}, inplace = True)

In [54]:
raw_accX = wavfile.read(r'/home/maxinehe/Downloads/'+fhex+'/acceleration_X.wav')
raw_accX = pd.DataFrame(data = raw_accX[1])
accx = 1.0000 * raw_accX 

In [55]:
raw_accY = wavfile.read(r'/home/maxinehe/Downloads/'+fhex+'/acceleration_Y.wav')
raw_accY = pd.DataFrame(data = raw_accY[1])
accy = 1.0000 * raw_accY 

In [56]:
raw_accZ = wavfile.read(r'/home/maxinehe/Downloads/'+fhex+'/acceleration_Z.wav')
raw_accZ = pd.DataFrame(data = raw_accZ[1])
accz = 1.0000 * raw_accZ 

In [57]:
raw_acc = pd.concat([accx, accy, accz], axis=1, ignore_index=True)
raw_acc.rename(columns = {0: 'ACC_X', 1: 'ACC_Y', 2: 'ACC_Z'}, inplace = True)
raw_acc

,ACC_X,ACC_Y,ACC_Z
0,152.0,-149.0,-186.0
1,151.0,-169.0,-178.0
2,117.0,-179.0,-149.0
3,85.0,-170.0,-130.0
4,71.0,-149.0,-133.0
...,...,...,...
531323,-92.0,-4.0,-148.0
531324,50.0,-330.0,-143.0
531325,147.0,-282.0,-185.0
531326,5.0,-140.0,-125.0


## Add timestamp to Hex (ECG & ACC) signal

In [58]:
t0_ecg = list(date_info.values())[0]/256
ecg['Timestamp'] = list(range(0, len(raw_ECG),1))
ecg['Timestamp'] = ecg['Timestamp'].apply(lambda x: x/256+t0_ecg)
#ecg['Timestamp'] = ecg['Timestamp'].str.get(0)
ecg['Second'] = ecg['Timestamp']
ecg = ecg.set_index('Timestamp')
ecg['Second'] = ecg['Second'].apply(lambda x: x-ecg.index[0])
ecg = ecg.reset_index()
#ecg.columns = ['Heart rate', 'Second']

In [59]:
ecg

,Timestamp,ECG,Second
0,1.713644e+09,19.0208,0.000000
1,1.713644e+09,26.2080,0.003906
2,1.713644e+09,26.2080,0.007812
3,1.713644e+09,26.2080,0.011719
4,1.713644e+09,26.2080,0.015625
...,...,...,...
2125323,1.713652e+09,8.7296,8302.042969
2125324,1.713652e+09,8.7296,8302.046875
2125325,1.713652e+09,8.7360,8302.050781
2125326,1.713652e+09,8.7488,8302.054688


In [60]:
t0_br = list(date_info.values())[0]/256
br['Timestamp'] = list(range(0, len(raw_br),1))
br['Timestamp'] = br['Timestamp'].apply(lambda x: x/1+t0_br)
br['Second'] = br['Timestamp']
br = br.set_index('Timestamp')
br['Second'] = br['Second'].apply(lambda x: x-br.index[0])
br = br.reset_index()

In [61]:
br

,Timestamp,breathing_rate,Second
0,1.713644e+09,10.0,0.0
1,1.713644e+09,10.0,1.0
2,1.713644e+09,10.0,2.0
3,1.713644e+09,10.0,3.0
4,1.713644e+09,10.0,4.0
...,...,...,...
8297,1.713652e+09,5.0,8297.0
8298,1.713652e+09,5.0,8298.0
8299,1.713652e+09,5.0,8299.0
8300,1.713652e+09,5.0,8300.0


In [62]:
t0_acc = list(date_info.values())[0]/256
raw_acc['Timestamp'] = list(range(0, len(raw_accX),1))
raw_acc['Timestamp'] = raw_acc['Timestamp'].apply(lambda x: x/64+t0_acc)
raw_acc['Second'] = raw_acc['Timestamp']
raw_acc = raw_acc.set_index('Timestamp')
raw_acc['Second'] = raw_acc['Second'].apply(lambda x: x-raw_acc.index[0])
raw_acc = raw_acc.reset_index()
raw_acc

,Timestamp,ACC_X,ACC_Y,ACC_Z,Second
0,1.713644e+09,152.0,-149.0,-186.0,0.000000
1,1.713644e+09,151.0,-169.0,-178.0,0.015625
2,1.713644e+09,117.0,-179.0,-149.0,0.031250
3,1.713644e+09,85.0,-170.0,-130.0,0.046875
4,1.713644e+09,71.0,-149.0,-133.0,0.062500
...,...,...,...,...,...
531323,1.713652e+09,-92.0,-4.0,-148.0,8301.921875
531324,1.713652e+09,50.0,-330.0,-143.0,8301.937500
531325,1.713652e+09,147.0,-282.0,-185.0,8301.953125
531326,1.713652e+09,5.0,-140.0,-125.0,8301.968750


# Synchronization for E4

In [63]:
def find_closest_index(timestamps, target):

    idx = (np.abs(timestamps - target)).argmin()
    return idx

In [64]:
def internalsync_offset(eda, temp, bvp, acc):
    # Extract timestamps to numpy arrays for efficient operations
    ts_eda = eda['Timestamp'].values
    ts_temp = temp['Timestamp'].values
    ts_bvp = bvp['Timestamp'].values
    ts_acc = acc['Timestamp'].values
    
    # Determine the common time frame across all signals
    start_time = max(ts_eda[0], ts_temp[0], ts_bvp[0], ts_acc[0])
    end_time = min(ts_eda[-1], ts_temp[-1], ts_bvp[-1], ts_acc[-1])
    
    # Synchronize each signal to the common time frame
    offsets = {}
    for name, ts in [('eda', ts_eda), ('temp', ts_temp), ('bvp', ts_bvp), ('acc', ts_acc)]:
        start_idx = find_closest_index(ts, start_time)
        end_idx = find_closest_index(ts, end_time) + 1  # Include the end index in the slice
        
        if name == 'eda':
            offsets['eda'] = eda.iloc[start_idx:end_idx]
        elif name == 'temp':
            offsets['temp'] = temp.iloc[start_idx:end_idx]
        elif name == 'bvp':
            offsets['bvp'] = bvp.iloc[start_idx:end_idx]
        elif name == 'acc':
            offsets['acc'] = acc.iloc[start_idx:end_idx]
    
    return offsets['eda'], offsets['temp'], offsets['bvp'], offsets['acc']

In [65]:
#accsync function is identical to doublesync_offset function
offset_eda, offset_temp, offset_bvp, offset_acc_e4 = internalsync_offset(a['EDA'],a['TEMP'],a['BVP'],a['ACC'])

In [66]:
offset_eda

,Timestamp,EDA,Second
0,1.713647e+09,0.000000,0.00
1,1.713647e+09,0.043563,0.25
2,1.713647e+09,0.088407,0.50
3,1.713647e+09,0.085844,0.75
4,1.713647e+09,0.085844,1.00
...,...,...,...
19507,1.713652e+09,0.067906,4876.75
19508,1.713652e+09,0.069188,4877.00
19509,1.713652e+09,0.070469,4877.25
19510,1.713652e+09,0.071750,4877.50


In [67]:
offset_temp

,Timestamp,Temp,Second
0,1.713647e+09,30.91,0.00
1,1.713647e+09,30.91,0.25
2,1.713647e+09,30.91,0.50
3,1.713647e+09,30.91,0.75
4,1.713647e+09,30.91,1.00
...,...,...,...
19507,1.713652e+09,27.41,4876.75
19508,1.713652e+09,27.39,4877.00
19509,1.713652e+09,27.39,4877.25
19510,1.713652e+09,27.39,4877.50


In [68]:
offset_bvp

,Timestamp,BVP,Second
0,1.713647e+09,-0.00,0.00
1,1.713647e+09,-0.00,0.02
2,1.713647e+09,-0.00,0.03
3,1.713647e+09,-0.00,0.05
4,1.713647e+09,-0.00,0.06
...,...,...,...
312172,1.713652e+09,-2.87,4877.69
312173,1.713652e+09,-2.68,4877.70
312174,1.713652e+09,-2.34,4877.72
312175,1.713652e+09,-1.88,4877.73


In [69]:
offset_acc_e4

,Timestamp,ACC_X,ACC_Y,ACC_Z,Second
0,1.713647e+09,-27.0,23.0,51.0,0.00000
1,1.713647e+09,-24.0,24.0,50.0,0.03125
2,1.713647e+09,-21.0,26.0,50.0,0.06250
3,1.713647e+09,-21.0,25.0,51.0,0.09375
4,1.713647e+09,-23.0,24.0,54.0,0.12500
...,...,...,...,...,...
156084,1.713652e+09,-49.0,-4.0,39.0,4877.62500
156085,1.713652e+09,-48.0,-4.0,39.0,4877.65625
156086,1.713652e+09,-48.0,-4.0,41.0,4877.68750
156087,1.713652e+09,-47.0,-4.0,42.0,4877.71875


# Export E4_CSV

In [70]:
offset_eda.to_csv('/home/maxinehe/Desktop/'+fE4+'_EDA.csv', sep=str(','), header=True)
offset_temp.to_csv('/home/maxinehe/Desktop/'+fE4+'_TEMP.csv', sep=str(','), header=True)
offset_bvp.to_csv('/home/maxinehe/Desktop/'+fE4+'_BVP.csv', sep=str(','), header=True)
offset_acc_e4.to_csv('/home/maxinehe/Desktop/'+fE4+'_acc.csv', sep=str(','), header=True)

# Pickling for E4

In [71]:
E4_to_dic = {}
E4_to_dic["EDA"] = offset_eda
E4_to_dic["TEMP"] = offset_temp
E4_to_dic["BVP"] = offset_bvp
E4_to_dic["Acceleration"] = offset_acc_e4
string = fE4 + '_E4.pickle'

with open(string, 'wb') as handle:
    pickle.dump(E4_to_dic, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
# with open('A04BA8_220801-180621_E4.pickle', 'rb') as handle:
#     b = pickle.load(handle)
# b

# Synchronization for Hex

In [72]:
offset_ecg, offset_br, offset_acc_hex, offset_acc_hex = internalsync_offset(ecg,br,raw_acc,raw_acc)

In [73]:
offset_acc_hex

,Timestamp,ACC_X,ACC_Y,ACC_Z,Second
0,1.713644e+09,152.0,-149.0,-186.0,0.000000
1,1.713644e+09,151.0,-169.0,-178.0,0.015625
2,1.713644e+09,117.0,-179.0,-149.0,0.031250
3,1.713644e+09,85.0,-170.0,-130.0,0.046875
4,1.713644e+09,71.0,-149.0,-133.0,0.062500
...,...,...,...,...,...
531260,1.713652e+09,79.0,-156.0,-171.0,8300.937500
531261,1.713652e+09,82.0,-148.0,-149.0,8300.953125
531262,1.713652e+09,94.0,-154.0,-147.0,8300.968750
531263,1.713652e+09,128.0,-36.0,-152.0,8300.984375


In [74]:
offset_br

,Timestamp,breathing_rate,Second
0,1.713644e+09,10.0,0.0
1,1.713644e+09,10.0,1.0
2,1.713644e+09,10.0,2.0
3,1.713644e+09,10.0,3.0
4,1.713644e+09,10.0,4.0
...,...,...,...
8297,1.713652e+09,5.0,8297.0
8298,1.713652e+09,5.0,8298.0
8299,1.713652e+09,5.0,8299.0
8300,1.713652e+09,5.0,8300.0


In [75]:
offset_ecg

,Timestamp,ECG,Second
0,1.713644e+09,19.0208,0.000000
1,1.713644e+09,26.2080,0.003906
2,1.713644e+09,26.2080,0.007812
3,1.713644e+09,26.2080,0.011719
4,1.713644e+09,26.2080,0.015625
...,...,...,...
2125052,1.713652e+09,10.0032,8300.984375
2125053,1.713652e+09,9.9392,8300.988281
2125054,1.713652e+09,9.8240,8300.992188
2125055,1.713652e+09,9.7536,8300.996094


# Export Hex_CSV

In [76]:
offset_br.to_csv('/home/maxinehe/Desktop/'+fhex+'_br.csv', sep=str(','), header=True)
offset_ecg.to_csv('/home/maxinehe/Desktop/'+fhex+'_ECG.csv', sep=str(','), header=True)
offset_acc_hex.to_csv('/home/maxinehe/Desktop/'+fhex+'_acc.csv', sep=str(','), header=True)

# Pickling for Hex

In [77]:
E4_to_dic = {}
E4_to_dic["ECG"] = offset_ecg
E4_to_dic["BR"] = offset_br
E4_to_dic["Acceleration"] = offset_acc_hex
string2 = fhex+'_hex.pickle'
with open(string2, 'wb') as handle:
    pickle.dump(E4_to_dic, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
# with open('record_249726_hex.pickle', 'rb') as handle:
#     c = pickle.load(handle)
# c

# Synchronization for ECG & BVP

In [78]:
def doublesync_offset(ecg, bvp):
    ecg_timestamps = ecg['Timestamp'].values
    bvp_timestamps = bvp['Timestamp'].values
    
    t1_ecg, t0_ecg = ecg_timestamps[-1], ecg_timestamps[0]
    t1_bvp, t0_bvp = bvp_timestamps[-1], bvp_timestamps[0]
    
    # BVP starts earlier and ends earlier than ECG
    if t0_bvp < t0_ecg and t1_bvp < t1_ecg:
        idx_start_ecg = find_closest_index(t0_ecg, bvp_timestamps)
        idx_end_bvp = find_closest_index(t1_bvp, ecg_timestamps)
        offset_ecg = ecg.iloc[idx_start_ecg:]
        offset_bvp = bvp.iloc[:idx_end_bvp+1]

    # BVP starts later and ends later than ECG
    elif t0_bvp > t0_ecg and t1_bvp > t1_ecg:
        idx_start_bvp = find_closest_index(t0_bvp, ecg_timestamps)
        idx_end_ecg = find_closest_index(t1_ecg, bvp_timestamps)
        offset_ecg = ecg.iloc[:idx_end_ecg+1]
        offset_bvp = bvp.iloc[idx_start_bvp:]

    # BVP starts earlier but ends later than ECG
    elif t0_bvp < t0_ecg and t1_bvp > t1_ecg:
        idx_start_bvp = find_closest_index(t0_ecg, bvp_timestamps)
        idx_end_bvp = find_closest_index(t1_ecg, bvp_timestamps)
        offset_bvp = bvp.iloc[idx_start_bvp:idx_end_bvp+1]
        offset_ecg = ecg
        
    # BVP starts later but ends earlier than ECG
    elif t0_bvp > t0_ecg and t1_bvp < t1_ecg:
        idx_start_ecg = find_closest_index(t0_bvp, ecg_timestamps)
        idx_end_ecg = find_closest_index(t1_bvp, ecg_timestamps)
        offset_ecg = ecg.iloc[idx_start_ecg:idx_end_ecg+1]
        offset_bvp = bvp
    return offset_ecg, offset_bvp

In [79]:
offset_ecg_cross, offset_bvp_cross = doublesync_offset(ecg, a['BVP'])

In [80]:
offset_ecg_cross

,Timestamp,ECG,Second
749642,1.713647e+09,8.9152,2928.289062
749643,1.713647e+09,8.9024,2928.292969
749644,1.713647e+09,8.9216,2928.296875
749645,1.713647e+09,8.9216,2928.300781
749646,1.713647e+09,8.8960,2928.304688
...,...,...,...
1999543,1.713652e+09,8.6656,7810.714844
1999544,1.713652e+09,8.6784,7810.718750
1999545,1.713652e+09,8.9408,7810.722656
1999546,1.713652e+09,8.8128,7810.726562


In [81]:
offset_bvp_cross

,Timestamp,BVP,Second
0,1.713647e+09,-0.00,0.00
1,1.713647e+09,-0.00,0.02
2,1.713647e+09,-0.00,0.03
3,1.713647e+09,-0.00,0.05
4,1.713647e+09,-0.00,0.06
...,...,...,...
312472,1.713652e+09,-5.30,4882.38
312473,1.713652e+09,-3.50,4882.39
312474,1.713652e+09,-1.92,4882.41
312475,1.713652e+09,-0.61,4882.42


# Export CSV

In [82]:
offset_ecg_cross.to_csv('/home/maxinehe/Desktop/'+fhex+'_ecg_cross.csv', sep=str(','), header=True)
offset_bvp_cross.to_csv('/home/maxinehe/Desktop/'+fE4+'_bvp_cross.csv', sep=str(','), header=True)

# Pickling for ECG & BVP

In [83]:
E4_to_dic = {}
E4_to_dic["ECG"] = offset_ecg_cross
E4_to_dic["BVP"] = offset_bvp_cross
string3 = fhex+'_cross_ecg_bvp.pickle'

with open(string3, 'wb') as handle:
    pickle.dump(E4_to_dic, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
# with open('record_249726_ECG&BVP.pickle', 'rb') as handle:
#     d = pickle.load(handle)
# d

# Synchronization for ACC

In [84]:
cross_acc_hex, cross_acc_e4 = doublesync_offset(raw_acc, a['ACC'])

# Export acc_CSV

In [85]:
cross_acc_hex.to_csv('/home/maxinehe/Desktop/'+fhex+'_acc_cross.csv', sep=str(','), header=True)
cross_acc_e4.to_csv('/home/maxinehe/Desktop/'+fE4+'_acc_cross.csv', sep=str(','), header=True)

# Pickling for ACC

In [86]:
E4_to_dic = {}
E4_to_dic["acc_e4"] = cross_acc_e4
E4_to_dic["acc_hex"] = cross_acc_hex
string4 = fhex+'cross_accE4_accHex.pickle'

with open(string4, 'wb') as handle:
    pickle.dump(E4_to_dic, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
# with open('record_249726_accE4&accHex.pickle', 'rb') as handle:
#     e = pickle.load(handle)
# e